In [8]:
# 変化量の上位95%だけを抽出
import pandas as pd
import os

# 入力ファイル
input_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/vad_dis.csv"

# 出力ファイル
output_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/results/Δv-95.csv"

# 出力先が無ければ新規作成
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# CSV読み込み
df = pd.read_csv(input_csv)

filter_95 = df[df["delta_valence"] > 0.549]

result = filter_95[
    [
        "filename",
        "text",
        "valence",
        "arousal",
        "dominance",
        "delta_valence",
        "delta_arousal",
        "delta_dominance",
    ]
]

# 保存
result.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"保存完了: {output_csv}")
print(f"発話数: {len(result)}")

保存完了: /home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/results/Δv-95.csv
発話数: 14


In [9]:
df[df["delta_valence"] > 0.549]

,filename,text,valence,arousal,dominance,delta_valence,delta_arousal,delta_dominance
6,0015_L.wav,<笑>,0.88,0.77,0.74,0.55,0.61,0.60
28,0054_L.wav,<笑>,0.96,0.91,0.85,0.56,0.68,0.63
56,0116_L.wav,<笑>,0.94,0.84,0.81,0.58,0.58,0.52
59,0122_L.wav,(F へー),0.37,0.18,0.23,0.61,0.69,0.58
67,0135_L.wav,そうなんだ,0.35,0.02,0.15,0.63,0.83,0.66
131,0263_L.wav,送ってたんだ,0.33,0.20,0.23,0.56,0.63,0.54
148,0297_L.wav,あれは(D お),0.34,0.32,0.29,0.60,0.27,0.37
157,0312_L.wav,<笑>,0.83,0.68,0.65,0.62,0.25,0.14
158,0314_L.wav,これこの子この後どうなるんだろう,0.28,0.71,0.66,0.55,0.03,0.01
161,0318_L.wav,感じですよね,0.29,0.15,0.24,0.55,0.49,0.37


In [11]:
# パーセンタイル95%のLの発話
import pandas as pd
import os
import shutil

# ==========================
# パス
# ==========================
l_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/vad_dis.csv"
wav_dir = "/home/mitani/CSJ-emo-int_bunseki/ALL-WAV"
output_root = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/results/eval-v"

os.makedirs(output_root, exist_ok=True)

# ==========================
# CSV読込（Lのみ）
# ==========================
df_l = pd.read_csv(l_csv)

# 発話番号（CSVのインデックスではなく、0015_L.wav の「15」を指定）
change_idx = [
    15, 54, 116, 122, 135, 263, 297, 312, 314, 318, 337, 377, 500, 504
]

context = 3

# ==========================
# 抽出
# ==========================
for utt_no in change_idx:

    # 発話番号からファイル名を作成
    target_filename = f"{utt_no:04d}_L.wav"

    # filenameから該当行を検索
    matched = df_l[df_l["filename"] == target_filename]

    if matched.empty:
        print(f"{target_filename} がCSV内に見つかりません")
        continue

    # DataFrame内でのインデックス取得
    idx = matched.index[0]

    print(f"発話番号 {utt_no} -> DataFrame index {idx}")

    # 前後3発話（Lのみ）
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Utterance number : {utt_no}\n")
        f.write(f"DataFrame index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 音声コピー
    for _, row in context_df.iterrows():

        src = os.path.join(wav_dir, row["filename"])

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # ターゲット音声にはTARGET_を付ける
        if row["filename"] == target_filename:
            dst_name = "TARGET_" + row["filename"]
        else:
            dst_name = row["filename"]

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

発話番号 15 -> DataFrame index 6
発話番号 54 -> DataFrame index 28
発話番号 116 -> DataFrame index 56
発話番号 122 -> DataFrame index 59
発話番号 135 -> DataFrame index 67
発話番号 263 -> DataFrame index 131
発話番号 297 -> DataFrame index 148
発話番号 312 -> DataFrame index 157
発話番号 314 -> DataFrame index 158
発話番号 318 -> DataFrame index 161
発話番号 337 -> DataFrame index 170
発話番号 377 -> DataFrame index 187
発話番号 500 -> DataFrame index 247
発話番号 504 -> DataFrame index 250
完了
